In [ ]:
# Minimal Cell G — mount vendor and proactively pin tokenizers==0.21.0 (no waiting)
import os, sys, shutil, subprocess, importlib, traceback

# exact, fixed config
DATASET_MOUNT_SLUG = "vendor-audioldm"
VENDOR_DIRNAME = "vendor_audioldm"
SOURCE = f"/kaggle/input/{DATASET_MOUNT_SLUG}"
DEST = "/kaggle/working/vendor_audioldm"
PIN_TOKENIZERS = "tokenizers==0.21.0"

# 1) copy vendor if missing
if not os.path.exists(DEST):
    if not os.path.exists(SOURCE):
        raise FileNotFoundError(f"Vendor dataset not mounted at {SOURCE}. Attach dataset and re-run.")
    shutil.copytree(SOURCE, DEST, dirs_exist_ok=True)

# 2) ensure vendor is first in import path
if DEST in sys.path:
    sys.path.remove(DEST)
sys.path.insert(0, DEST)

# 3) install pinned tokenizers into vendor (idempotent)
ret = subprocess.call([sys.executable, "-m", "pip", "install", "--target", DEST, PIN_TOKENIZERS, "--no-deps", "--upgrade"])
if ret != 0:
    raise RuntimeError(f"Failed to pip install {PIN_TOKENIZERS} into vendor (return code {ret})")

# 4) unload any previously loaded relevant modules so re-import uses vendor
for prefix in ("transformers", "tokenizers", "diffusers", "huggingface_hub", "safetensors"):
    for mod in list(sys.modules):
        if mod == prefix or mod.startswith(prefix + "."):
            sys.modules.pop(mod, None)

# 5) import and verify
try:
    transformers = importlib.import_module("transformers")
    diffusers = importlib.import_module("diffusers")
    tokenizers = importlib.import_module("tokenizers")
    print("Imported from vendor:", "transformers", transformers.__version__,
          "diffusers", diffusers.__version__, "tokenizers", tokenizers.__version__)
except Exception:
    print("Import failed; full traceback follows:")
    traceback.print_exc()
    raise

print("Vendor mounted and tokenizers pinned at", DEST)

In [ ]:
# ---------------------------------------------------------------------------
# CONFIGURATION CELL
# ---------------------------------------------------------------------------
# This notebook replaces SDXL-ControlNet T2I with AudioLDM2 text-to-audio
# You can change AUDIO_LENGTH_S and DEVICE here.
AUDIO_LENGTH_S = 5.0  # default generated audio length in seconds
SAMPLE_RATE = 16000   # output sample rate (Hz)
MODEL_DATASET_NAME = "audioldm2"  # assumed kaggle dataset folder under /kaggle/input/
USE_FP16 = True
DEFAULT_REPO_ID = "cvssp/audioldm2"  # HF repo to use if dataset not mounted


In [ ]:
import os, sys

p = "/kaggle/input/sage-zrok-token/.zrok_api_key"
zrok_token = None

if os.path.isfile(p):
    with open(p, "r", encoding="utf-8", errors="ignore") as f:
        zrok_token = f.read().strip()

if not zrok_token:
    print("❌ Token not found or empty:", p)
    # Not exiting automatically in case you want to run locally without zrok
    # sys.exit(1)
else:
    print("✓ Found zrok token file")

In [ ]:
print("Checking for local AudioLDM2 dataset at /kaggle/input/" + MODEL_DATASET_NAME)
local_model_path = f"/kaggle/input/{MODEL_DATASET_NAME}"
    
if os.path.exists(local_model_path):
    print(f"✓ Found dataset folder: {local_model_path} — will attempt to load model from there")
else:
    print(f"ℹ️ Dataset not found at {local_model_path}. Will load from HF repo: {DEFAULT_REPO_ID} (diffusers will download)")

In [ ]:
import os
import torch
import numpy as np
from diffusers import AudioLDM2Pipeline
import scipy.io.wavfile

# --- Device ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# --- Load pipeline (either from local dataset or HF repo) ---
repo_or_path = local_model_path if os.path.exists(local_model_path) else DEFAULT_REPO_ID

print(f"Loading AudioLDM2 pipeline from: {repo_or_path} (this may download weights if not local)")

# Use float16 on CUDA for memory savings
torch_dtype = torch.float16 if (USE_FP16 and device == "cuda") else torch.float32

try:
    pipe = AudioLDM2Pipeline.from_pretrained(repo_or_path, torch_dtype=torch_dtype)
    pipe = pipe.to(device)
    print("✓ AudioLDM2 pipeline loaded and moved to device")
except Exception as e:
    print("Failed to load pipeline from", repo_or_path)
    print(e)
    raise

# You can adjust scheduler or other internals here if desired


In [ ]:
from fastapi import FastAPI
import nest_asyncio
import uvicorn

app = FastAPI()
# allow running uvicorn inside the notebook
nest_asyncio.apply()


In [ ]:
# ===================================================================
# ASYNCHRONOUS TASK HANDLING SETUP
# ===================================================================
from fastapi import BackgroundTasks, HTTPException, Response
import uuid
import io
import time
import traceback

# In-memory tasks dictionary
# task_id -> { status: 'PENDING'|'PROCESSING'|'COMPLETED'|'FAILED', result: bytes, error: str }
tasks = {}


In [ ]:
def _convert_audio_to_wav_bytes(audio_np: np.ndarray, sample_rate: int = SAMPLE_RATE) -> bytes:
    """
    Convert a numpy audio array to 16-bit PCM WAV bytes.
    Accepts float32 arrays in [-1,1] or int16 arrays.
    If multi-channel, expects shape (n_samples, n_channels).
    """
    # Ensure numpy
    audio = np.asarray(audio_np)

    # If shape is (channels, samples), convert to (samples, channels)
    if audio.ndim == 2 and audio.shape[0] > audio.shape[1]:
        # Heuristic: maybe channels first -> transpose
        audio = audio.T

    # If float, convert to int16
    if np.issubdtype(audio.dtype, np.floating):
        # clip
        audio = np.clip(audio, -1.0, 1.0)
        int16 = (audio * 32767).astype(np.int16)
    elif np.issubdtype(audio.dtype, np.integer):
        # if already int16, leave
        if audio.dtype != np.int16:
            # scale/rescale conservatively
            max_val = np.max(np.abs(audio))
            if max_val == 0:
                int16 = audio.astype(np.int16)
            else:
                scale = 32767.0 / max_val
                int16 = (audio * scale).astype(np.int16)
        else:
            int16 = audio
    else:
        # fallback
        int16 = audio.astype(np.int16)

    # Write to wav bytes using scipy
    out_buffer = io.BytesIO()
    # scipy expects shape (n_samples, n_channels) or (n_samples,) for mono
    scipy.io.wavfile.write(out_buffer, sample_rate, int16)
    out_buffer.seek(0)
    return out_buffer.read()


In [ ]:
def run_audio_generation_task(task_id: str, params: dict):
    """
    Background worker to generate audio and update tasks dict.
    params expected keys:
      - prompt
      - seed (optional)
      - num_inference_steps
      - guidance_scale (optional)
      - audio_length_in_s
    """
    try:
        tasks[task_id]["status"] = "PROCESSING"

        prompt = params.get("prompt")
        seed = params.get("seed")
        num_inference_steps = params.get("num_inference_steps", 200)
        audio_length_in_s = params.get("audio_length_in_s", AUDIO_LENGTH_S)
        guidance_scale = params.get("guidance_scale", None)

        # Prepare generator / seed
        generator = None
        if seed is None:
            seed = int(torch.randint(0, 2**31 - 1, (1,)).item())
        if device == "cuda":
            generator = torch.Generator(device="cuda").manual_seed(seed)
        else:
            generator = torch.Generator(device="cpu").manual_seed(seed)

        # Call pipeline
        # The AudioLDM2 pipeline signature: pipe(prompt, num_inference_steps=..., audio_length_in_s=..., generator=...)
        # Some versions accept guidance_scale, some don't. We'll pass guidance via kwargs if accepted.
        call_kwargs = {
            "prompt": prompt,
            "num_inference_steps": int(num_inference_steps),
            "audio_length_in_s": float(audio_length_in_s),
            "generator": generator,
        }
        if guidance_scale is not None:
            call_kwargs["guidance_scale"] = float(guidance_scale)

        # Run pipeline (it returns a PipelineOutput with .audios)
        out = pipe(**call_kwargs)
        audio = out.audios[0]

        # Convert to wav bytes
        wav_bytes = _convert_audio_to_wav_bytes(audio, sample_rate=SAMPLE_RATE)

        tasks[task_id]["status"] = "COMPLETED"
        tasks[task_id]["result"] = wav_bytes
        tasks[task_id]["meta"] = {
            "prompt": prompt,
            "seed": seed,
            "audio_length_in_s": audio_length_in_s,
            "sample_rate": SAMPLE_RATE,
            "num_inference_steps": num_inference_steps,
        }

    except Exception as e:
        print(f"--- ASYNC AUDIO TASK {task_id} FAILED ---")
        traceback.print_exc()
        tasks[task_id]["status"] = "FAILED"
        tasks[task_id]["error"] = str(e)


In [ ]:
from fastapi import Form, UploadFile, File
from fastapi.responses import StreamingResponse, JSONResponse

@app.post("/generate")
async def generate(
    prompt: str = Form(...),
    seed: int = Form(None),
    num_inference_steps: int = Form(200),
    audio_length_in_s: float = Form(AUDIO_LENGTH_S),
    guidance_scale: float = Form(None),
):
    """
    Synchronous generation: returns WAV audio bytes directly.
    """
    try:
        # Prepare generator
        if seed is None:
            seed = int(torch.randint(0, 2**31 - 1, (1,)).item())
        if device == "cuda":
            generator = torch.Generator(device="cuda").manual_seed(seed)
        else:
            generator = torch.Generator(device="cpu").manual_seed(seed)

        call_kwargs = {
            "prompt": prompt,
            "num_inference_steps": int(num_inference_steps),
            "audio_length_in_s": float(audio_length_in_s),
            "generator": generator,
        }
        if guidance_scale is not None:
            call_kwargs["guidance_scale"] = float(guidance_scale)

        out = pipe(**call_kwargs)
        audio = out.audios[0]
        wav_bytes = _convert_audio_to_wav_bytes(audio, sample_rate=SAMPLE_RATE)

        return StreamingResponse(io.BytesIO(wav_bytes), media_type="audio/wav")

    except Exception as e:
        traceback.print_exc()
        return JSONResponse(status_code=500, content={"error": str(e), "traceback": traceback.format_exc()})


In [ ]:
@app.post("/generate-async")
    
async def generate_async(
    background_tasks: BackgroundTasks,
    prompt: str = Form(...),
    seed: int = Form(None),
    num_inference_steps: int = Form(200),
    audio_length_in_s: float = Form(AUDIO_LENGTH_S),
    guidance_scale: float = Form(None),
):
    """
    Start audio generation in background and return a task_id for polling.
    """
    task_id = str(uuid.uuid4())
    tasks[task_id] = {"status": "PENDING"}

    params = {
        "prompt": prompt,
        "seed": seed,
        "num_inference_steps": num_inference_steps,
        "audio_length_in_s": audio_length_in_s,
        "guidance_scale": guidance_scale,
    }

    background_tasks.add_task(run_audio_generation_task, task_id, params)
    return {"task_id": task_id, "status": "PENDING"}


In [ ]:
@app.get("/status/{task_id}")
async def get_status(task_id: str):
    """
    Poll the status of a background generation task. If completed, return WAV bytes.
    """
    task = tasks.get(task_id)
    if not task:
        raise HTTPException(status_code=404, detail="Task not found")

    status = task.get("status")
    if status == "COMPLETED":
        wav_bytes = task.get("result")
        # Optionally remove the task to free memory
        del tasks[task_id]
        return Response(content=wav_bytes, media_type="audio/wav")
    elif status == "FAILED":
        error_message = task.get("error", "Unknown error")
        del tasks[task_id]
        raise HTTPException(status_code=500, detail=f"Task failed: {error_message}")
    else:
        return {"task_id": task_id, "status": status}


In [ ]:
# Download zrok v1.1.3 (or use already present binary if you uploaded one)
!wget -q https://github.com/openziti/zrok/releases/download/v1.1.3/zrok_1.1.3_linux_amd64.tar.gz || true
!tar -xzf zrok_1.1.3_linux_amd64.tar.gz || true
!chmod +x zrok || true

# Enable headless (will use $zrok_token environment if present)
import subprocess, shlex, os
if zrok_token:
    print('Enabling zrok in headless mode...')
    # write token to env for zrok command
    os.environ['ZROK_API_KEY'] = zrok_token
    # if zrok supports --headless with token env, call directly
    try:
        subprocess.run(["./zrok", "enable", "--headless", zrok_token], check=True)
        print('zrok enable finished')
    except Exception as e:
        print('zrok enable error (continuing):', e)
else:
    print('No zrok token available in notebook environment; skipping zrok enable')


In [ ]:
import threading

import uvicorn

def run_uvicorn():
    # note: use log_level="info" to match your previous setup
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

threading.Thread(target=run_uvicorn, daemon=True).start()
print("uvicorn started in background thread")

In [ ]:
import subprocess
import time


def start_zrok_tunnel(port=8000):
    # Start the tunnel
    process = subprocess.Popen([
        "./zrok", "share", "public", f"localhost:{port}", "--headless"
    ], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

    # Give it a moment to start
    time.sleep(3)

    # Check agent status to get the URL
    status_process = subprocess.run([
        "./zrok", "agent", "status"
    ], capture_output=True, text=True)

    print("Agent Status:")
    print(status_process.stdout)

    return process

# Only start zrok if token is present
if zrok_token:
    tunnel_process = start_zrok_tunnel(8000)
    print("Zrok tunnel started! Check the agent status above for your public URL.")
else:
    print("Skipping zrok tunnel start because token was not found.")

In [ ]:
print("Server and zrok tunnel (if enabled) are running. Keeping the notebook alive...")

try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("Shutting down.")